In [2]:
# =========================================================
# Setup
# =========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18
from torch.utils.data import DataLoader
import numpy as np
import random
import pandas as pd
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

torch.backends.cudnn.benchmark = True

Device: cuda


In [3]:
# =========================================================
# Transforms
# =========================================================

class TwoCropsTransform:
    def __init__(self, base_transform):
        self.base_transform = base_transform

    def __call__(self, x):
        return self.base_transform(x), self.base_transform(x)


ssl_transform = TwoCropsTransform(T.Compose([
    T.RandomResizedCrop(96, scale=(0.2,1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4,0.4,0.4,0.1),
    T.RandomGrayscale(p=0.2),
    T.ToTensor()
]))

probe_transform = T.Compose([
    T.Resize(96),
    T.ToTensor()
])

In [4]:
# =========================================================
# Encoder
# =========================================================

class Encoder(nn.Module):

    def __init__(self):
        super().__init__()
        backbone = resnet18(weights=None)
        backbone.fc = nn.Identity()
        self.backbone = backbone

    def forward(self,x):
        return self.backbone(x)

In [5]:
class MLP(nn.Module):

    def __init__(self,in_dim,hidden_dim=2048,out_dim=256):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(in_dim,hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,out_dim)
        )

    def forward(self,x):
        return self.net(x)

In [6]:
class BYOL(nn.Module):

    def __init__(self):

        super().__init__()

        self.online_encoder = Encoder()
        self.online_projector = MLP(512)
        self.online_predictor = MLP(256,512,256)

        self.target_encoder = Encoder()
        self.target_projector = MLP(512)

        for p in self.target_encoder.parameters():
            p.requires_grad=False

        for p in self.target_projector.parameters():
            p.requires_grad=False


    @torch.no_grad()
    def update_target(self,tau=0.996):

        for online,target in zip(
            self.online_encoder.parameters(),
            self.target_encoder.parameters()
        ):
            target.data = tau*target.data + (1-tau)*online.data


    def forward(self,x1,x2):

        o1 = self.online_predictor(
            self.online_projector(self.online_encoder(x1))
        )

        with torch.no_grad():
            t2 = self.target_projector(
                self.target_encoder(x2)
            )

        loss = 2 - 2*F.cosine_similarity(o1,t2.detach(),dim=-1)

        return loss.mean()

In [7]:
class SimCLR(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = Encoder()
        self.projector = MLP(512)


    def forward(self,x):

        z = self.projector(self.encoder(x))

        return F.normalize(z,dim=1)


def nt_xent_loss(z1,z2,tau=0.5):

    N = z1.size(0)

    z = torch.cat([z1,z2],dim=0)

    sim = torch.mm(z,z.t()) / tau

    mask = torch.eye(2*N,device=z.device).bool()

    sim.masked_fill_(mask,-9e15)

    positives = torch.cat([
        torch.arange(N,2*N),
        torch.arange(0,N)
    ]).to(z.device)

    return F.cross_entropy(sim,positives)

In [8]:
def jacobian_reg(model,x):

    x = x.clone().detach().requires_grad_(True)

    y = model.encoder(x)

    v = torch.randn_like(y)

    Jv = torch.autograd.grad(
        outputs=y,
        inputs=x,
        grad_outputs=v,
        retain_graph=True
    )[0]

    return torch.sqrt(Jv.pow(2).sum()/x.size(0))

In [9]:
class LinearProbe(nn.Module):

    def __init__(self,encoder):

        super().__init__()

        self.encoder = encoder

        for p in self.encoder.parameters():
            p.requires_grad=False

        self.fc = nn.Linear(512,10)


    def forward(self,x):

        with torch.no_grad():
            feat = self.encoder(x)

        return self.fc(feat)

In [10]:
def train_probe(encoder,train_loader,test_loader,epochs=10):

    model = LinearProbe(encoder).to(device)

    optimizer = torch.optim.Adam(model.fc.parameters(),lr=1e-3)

    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):

        model.train()

        for x,y in train_loader:

            x,y = x.to(device),y.to(device)

            loss = criterion(model(x),y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()


    model.eval()

    correct,total = 0,0

    with torch.no_grad():

        for x,y in test_loader:

            x,y = x.to(device),y.to(device)

            preds = model(x).argmax(1)

            correct += (preds==y).sum().item()

            total += y.size(0)

    return correct/total

In [11]:
# =========================================================
# STL10 (FAST SUBSET FOR SSL TRAINING)
# =========================================================

ssl_dataset_full = torchvision.datasets.STL10(
    root="./data",
    split="unlabeled",
    download=True,
    transform=ssl_transform
)

subset_size = 20000

indices = torch.randperm(len(ssl_dataset_full))[:subset_size]

ssl_dataset = torch.utils.data.Subset(
    ssl_dataset_full,
    indices
)

ssl_loader = DataLoader(
    ssl_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)


# =========================================================
# CIFAR10 (linear probe dataset)
# =========================================================

cifar_train = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=probe_transform
)

cifar_test = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=probe_transform
)

cifar_train_loader = DataLoader(
    cifar_train,
    batch_size=128,
    shuffle=True,
    num_workers=2
)

cifar_test_loader = DataLoader(
    cifar_test,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

print("SSL subset size:", subset_size)

100%|██████████| 2.64G/2.64G [02:48<00:00, 15.7MB/s] 
100%|██████████| 170M/170M [00:02<00:00, 84.1MB/s] 


SSL subset size: 20000


In [12]:
# =========================================================
# FAST CROSS DATASET TRANSFER
# STL10 → CIFAR10
# =========================================================

ssl_epochs = 100
probe_seeds = [0,1,2]

results = []


# =========================================================
# SSL TRAIN FUNCTION
# =========================================================

def train_ssl(model, method_name, use_jac=False):

    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

    print("\nTraining:", method_name)

    total_start = time.time()

    for epoch in range(ssl_epochs):

        epoch_start = time.time()

        model.train()

        for (x1,x2),_ in ssl_loader:

            x1,x2 = x1.to(device),x2.to(device)

            if method_name == "BYOL":

                loss = model(x1,x2)

            else:

                z1 = model(x1)
                z2 = model(x2)

                loss = nt_xent_loss(z1,z2)

                if use_jac:

                    jac = jacobian_reg(model,x1)

                    loss = loss + 0.01 * jac


            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if method_name == "BYOL":

                model.update_target()


        epoch_time = time.time() - epoch_start

        print(f"Epoch {epoch+1:02d}/{ssl_epochs} | {epoch_time:.1f}s")


    total_time = (time.time() - total_start) / 60

    print(f"{method_name} training finished in {total_time:.2f} minutes")

    return model


# =========================================================
# TRAIN SSL ENCODERS
# =========================================================

byol = train_ssl(BYOL(), "BYOL")

simclr = train_ssl(SimCLR(), "SimCLR")

simclr_jac = train_ssl(SimCLR(), "SimCLR+Jac", use_jac=True)


# =========================================================
# PROBE SEEDS
# =========================================================

for seed in probe_seeds:

    print("\nProbe Seed:", seed)

    set_seed(seed)


    byol_acc = train_probe(
        byol.online_encoder,
        cifar_train_loader,
        cifar_test_loader
    )

    print("BYOL Transfer Acc:", byol_acc)


    simclr_acc = train_probe(
        simclr.encoder,
        cifar_train_loader,
        cifar_test_loader
    )

    print("SimCLR Transfer Acc:", simclr_acc)


    jac_acc = train_probe(
        simclr_jac.encoder,
        cifar_train_loader,
        cifar_test_loader
    )

    print("SimCLR+Jac Transfer Acc:", jac_acc)


    results.append({
        "Seed": seed,
        "BYOL_Transfer": byol_acc,
        "SimCLR_Transfer": simclr_acc,
        "SimCLRJac_Transfer": jac_acc
    })


    pd.DataFrame(results).to_csv(
        "cross_dataset_transfer.csv",
        index=False
    )


Training: BYOL
Epoch 01/100 | 51.0s
Epoch 02/100 | 45.2s
Epoch 03/100 | 44.9s
Epoch 04/100 | 44.9s
Epoch 05/100 | 44.7s
Epoch 06/100 | 44.4s
Epoch 07/100 | 45.2s
Epoch 08/100 | 46.0s
Epoch 09/100 | 45.3s
Epoch 10/100 | 45.7s
Epoch 11/100 | 45.4s
Epoch 12/100 | 44.7s
Epoch 13/100 | 44.8s
Epoch 14/100 | 45.3s
Epoch 15/100 | 45.3s
Epoch 16/100 | 44.6s
Epoch 17/100 | 45.5s
Epoch 18/100 | 44.7s
Epoch 19/100 | 44.5s
Epoch 20/100 | 45.6s
Epoch 21/100 | 45.7s
Epoch 22/100 | 46.6s
Epoch 23/100 | 47.6s
Epoch 24/100 | 44.1s
Epoch 25/100 | 46.0s
Epoch 26/100 | 47.7s
Epoch 27/100 | 47.4s
Epoch 28/100 | 47.0s
Epoch 29/100 | 46.8s
Epoch 30/100 | 46.9s
Epoch 31/100 | 47.3s
Epoch 32/100 | 47.5s
Epoch 33/100 | 46.8s
Epoch 34/100 | 47.5s
Epoch 35/100 | 47.8s
Epoch 36/100 | 47.4s
Epoch 37/100 | 45.1s
Epoch 38/100 | 43.7s
Epoch 39/100 | 43.7s
Epoch 40/100 | 43.5s
Epoch 41/100 | 44.6s
Epoch 42/100 | 44.0s
Epoch 43/100 | 44.7s
Epoch 44/100 | 45.0s
Epoch 45/100 | 44.2s
Epoch 46/100 | 43.9s
Epoch 47/100 | 43.

In [13]:
# =========================================================
# ADDITIONAL PROBE SEEDS (3,4)
# =========================================================

extra_seeds = [3,4]

for seed in extra_seeds:

    print("\nProbe Seed:", seed)

    set_seed(seed)

    # BYOL probe
    byol_acc = train_probe(
        byol.online_encoder,
        cifar_train_loader,
        cifar_test_loader
    )

    print("BYOL Transfer Acc:", byol_acc)


    # SimCLR probe
    simclr_acc = train_probe(
        simclr.encoder,
        cifar_train_loader,
        cifar_test_loader
    )

    print("SimCLR Transfer Acc:", simclr_acc)


    # SimCLR + Jac probe
    jac_acc = train_probe(
        simclr_jac.encoder,
        cifar_train_loader,
        cifar_test_loader
    )

    print("SimCLR+Jac Transfer Acc:", jac_acc)


    results.append({
        "Seed": seed,
        "BYOL_Transfer": byol_acc,
        "SimCLR_Transfer": simclr_acc,
        "SimCLRJac_Transfer": jac_acc
    })


# =========================================================
# FINAL RESULTS TABLE
# =========================================================

df = pd.DataFrame(results)

print("\nFinal Results")
print(df)


# mean/std summary
summary = df[[
    "BYOL_Transfer",
    "SimCLR_Transfer",
    "SimCLRJac_Transfer"
]].agg(["mean","std"])

print("\nMean ± Std")
print(summary)


# =========================================================
# SAVE CSV TO KAGGLE WORKING DIRECTORY
# =========================================================

df.to_csv("/kaggle/working/cross_dataset_transfer_full.csv", index=False)

summary.to_csv("/kaggle/working/cross_dataset_transfer_summary.csv")

print("\nFiles saved:")
print("/kaggle/working/cross_dataset_transfer_full.csv")
print("/kaggle/working/cross_dataset_transfer_summary.csv")


Probe Seed: 3
BYOL Transfer Acc: 0.6525
SimCLR Transfer Acc: 0.6807
SimCLR+Jac Transfer Acc: 0.6827

Probe Seed: 4
BYOL Transfer Acc: 0.6422
SimCLR Transfer Acc: 0.682
SimCLR+Jac Transfer Acc: 0.68

Final Results
   Seed  BYOL_Transfer  SimCLR_Transfer  SimCLRJac_Transfer
0     0         0.6441           0.6870              0.6801
1     1         0.6462           0.6821              0.6842
2     2         0.6420           0.6826              0.6836
3     3         0.6525           0.6807              0.6827
4     4         0.6422           0.6820              0.6800

Mean ± Std
      BYOL_Transfer  SimCLR_Transfer  SimCLRJac_Transfer
mean       0.645400         0.682880            0.682120
std        0.004317         0.002408            0.001964

Files saved:
/kaggle/working/cross_dataset_transfer_full.csv
/kaggle/working/cross_dataset_transfer_summary.csv
